<a href="https://colab.research.google.com/github/betr0dalf/TIMO/blob/main/TIMO_NovikovDV_prac5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ?????????? ? ?????????????? ????????? ????????
# ???????????? ?????? ? 5
# ???????? ??????? ?????? ????-05-22
# ??????? ????? ?????????????

## ?????? ??????????? ?????????

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import tree
import xgboost as xgb
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, silhouette_score
from sklearn.metrics import silhouette_samples, precision_recall_curve, auc
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

## ??????? ?1:

????? ? ??????????? ?????? ??? ?????? ?????????????
*   ??????? ????? ?????? ??? ??????? ?????? ????????????? ??? ?????????. ?????? ?? ?????? ??????????? ????? ??????????? ??????.
*   ??????????? ?????? ??? ???????: ????????? ?? ?? ??????? ?????????, ???????? ? ??? ????????????? ??????????? ??????? ??????.

In [ ]:
# ???????? ??????
data = pd.read_csv('diabetes_dataset.csv')
# ???????? ??????
data.head()

In [ ]:
# ???????? ????????????? ?????
data = data.drop_duplicates()

# ???????????? ??? Min-Max Scaling
scaler = MinMaxScaler()
data =  pd.DataFrame(scaler.fit_transform(data), columns=data.columns)
print('??????????? ??????:', data.shape, end='\n\n')
data.head()

In [ ]:
# ??????????? ???????? ???????
x = data.drop("Outcome", axis=1)
y = data["Outcome"]

# ?????????? ?????? ?? ????????? ? ???????? ???????
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## ??????? ?2:

????????? ?????? ???????

*   ?????????? ?????? ??????? ??? ?????? ????????????? ??? ????????? ? ?????????????? ?????????? scikit-learn.
*   ????????? ???????? ?????? ?? ????? ??????.

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=3)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 2), '\n')
plt.figure(figsize=(12,8))
tree.plot_tree(clf, filled=True)
plt.show()

## ??????? ?3:

????????? ?????????????? ?????? ???????

In [ ]:
dt = DecisionTreeClassifier()

# ??????????? ????? ???????????????
param_grid = {
'max_depth': [2, 3, 4, 5, 6],
'min_samples_split': [2, 3, 4, 5],
'min_samples_leaf': [2, 3, 4, 5],
'criterion': ['entropy', 'gini']
}

# ????????? GridSearchCV
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='accuracy')
# ???????? ? ?????? ???????????????
grid_search.fit(X_train, y_train)
# ?????? ??????????????
print("?????? ?????????:", grid_search.best_params_)
# ?????? ?????? ? ??????? ???????????
best_model = grid_search.best_estimator_
print("???????? ?? ???????? ??????:", best_model.score(X_test, y_test))

In [ ]:
clf = tree.DecisionTreeClassifier(max_depth=5, min_samples_split=2, min_samples_leaf=2, criterion='gini')
clf.fit(X_train, y_train)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 2), '\n')
plt.figure(figsize=(12,8))
tree.plot_tree(clf, filled=True)
plt.show()

## ??????? ?4:

?????????? ????????? ???????? ???????:
*   ??????? (bagging)
*   ??????? (????????, ? ??????? XGBoost, CatBoost ??? LightGBM)
*   ??????? (stacking)
*   ??????? ???????? ?? ??? ?? ?????? ? ???????? ?? ? ???????????? ?????? ???????.

In [ ]:
# Bagging

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
# ?????? ????????
accuracy = rf.score(X_test, y_test)

In [ ]:
# XGBoost

xg_clf = xgb.XGBClassifier(n_estimators=100, seed=42)
xg_clf.fit(X_train, y_train)
# ?????? ????????
accuracy = xg_clf.score(X_test, y_test)

In [ ]:
# ??????????? ??????? ???????

base_estimators = [
('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
('dt', DecisionTreeClassifier(max_depth=3, random_state=42))
]
# ??????????
final_estimator = LogisticRegression()
# ???????? ???????-??????????????
stacking_clf = StackingClassifier(estimators=base_estimators, final_estimator=final_estimator)
# ???????? ? ?????? ??????
stacking_clf.fit(X_train, y_train)
accuracy = stacking_clf.score(X_test, y_test)

## ??????? ?5:

??????? ???????? ???????:
*   ??????? ???????? ????????? ??????? ? ??????????????
??????, ??????????? ???, ??? ?????????????? ??? ?????? ???????.
*   ???????? ?????????????????? ????????? ????????? ? ??????? ??????? ?????? ??????? ? ???????? ??????, ????? ?????? ????? ?????? ??????.

In [ ]:
def count_metric(X_test, y_test, model):
    # ???????????? ??????
    y_pred = model.predict(X_test)
    # ???????
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    print("????????:", accuracy)
    print("???????? (Precision):", precision)
    print("??????? (Recall):", recall)
    print("F1-score:", f1)
    # ??????? ??????
    cm = confusion_matrix(y_test, y_pred)
    print("??????? ??????:\n", cm)
    return {
          'Accuracy': accuracy,
          'Precision': precision,
          'Recall': recall,
          'F1-score': f1,
      }

In [ ]:
metrics={}
print('Decision Tree', end='\n\n')
metrics['Decision Tree'] = count_metric( X_test, y_test, clf)

In [ ]:
print('Random Forest Classifier', end='\n\n')
metrics['Random Forest Classifier'] = count_metric(X_test, y_test, rf)

In [ ]:
print('XG Classifier')
metrics['XG Classifier'] = count_metric( X_test, y_test, xg_clf)

In [ ]:
print('Stacking Classifier')
metrics['Stacking Classifier'] = count_metric(X_test, y_test, stacking_clf,)

In [ ]:
all_metrics = pd.DataFrame(metrics).T
display(all_metrics)